# Train and Evaluate AutoGluon Classification Models

<img src="../data/img/autogluon-bagging.png" width="600" height="200">

This notebook is a thin wrapper around `tools/evaluate_autogluon_classification.py`. It exposes the same inputs as the script's `__main__` section and runs the full modeling workflow from within Jupyter.


In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if p.name == 'pips-design-toolkit'), None)
if repo_root is None:
    raise RuntimeError('Run this notebook from within the pips-design-toolkit repository.')

tools_dir = repo_root / "tools"
os.chdir(tools_dir)
if str(tools_dir) not in sys.path:
    sys.path.insert(0, str(tools_dir))

from types import SimpleNamespace
import pandas as pd

from evaluate_autogluon_classification import (
    ml_autogluon_train_test_random,
    _default_ml_prediction_dir,
    _default_project_data_dir,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)


## Inputs

Edit the values in the next cell to match the run you want.

Key groups:
- `data_folder`, `project_data_folder`, `input_mode`, `train_input_fname`, `test_input_fname`, `labels_actual_fname`, `input_subfolder`, `output_subfolder`
- `n_splits`, `num_bag_folds`, `num_stack_levels`, `split_type`, `use_precomputed_folds`, `stratify`, `random_state`
- `filt_by`, `filt_shanms`, `filt_sift`, `filt_dist`, `filter_stage`
- `featureset`, `ylabel`, `data_fracs`
- `output_fname`, `save_model`, `load_model`, `save_y_predictions`, `show_only_test_results`

A few practical notes:
- `train_input_fname` controls the label filename default and the prefix used to find feature files.
- `input_subfolder` controls where assembled feature files are searched.
- `output_subfolder` controls where outputs are saved under `data/ml_prediction/Output/`.
- Use `input_mode='assembled'` to build the dataset from labels plus component feature files.
- Use `input_mode='precompiled'` to load an already-compiled CSV from `data/ml_prediction/Input/`.


In [ ]:
args = SimpleNamespace(
    data_folder=_default_ml_prediction_dir(),
    project_data_folder=_default_project_data_dir(),
    input_mode='assembled',
    train_input_fname='GOh1052',
    test_input_fname=None,  # e.g. 'GOh1052_TEST-reduced'
    labels_actual_fname='',
    input_subfolder='',
    output_subfolder='GOh1052',
    n_splits=4,
    num_bag_folds=8,
    num_stack_levels=1,
    filt_by='',
    filt_shanms=[0.5, 1.5],
    filt_sift=[0.1, 0.45],
    filt_dist=[20, 50],
    featureset='Aggregation',
    ylabel='CategoryV3',
    output_fname='GOh1052_Aggregation',
    save_model='../data/ml_prediction/AutogluonModels',
    load_model=None,  # e.g. '../data/ml_prediction/AutogluonModels/'
    split_type='random',  # or 'custom'
    filter_stage='train_only',
    use_precomputed_folds=False,
    stratify=True,
    random_state=42,
    n_boot=1000,
    ci=0.95,
    data_fracs=[1.0],
    save_y_predictions=True,
    show_only_test_results=False,
)

pd.DataFrame(sorted(vars(args).items()), columns=['argument', 'value'])


## Run the workflow

Executing the next cell runs the same end-to-end modeling workflow as the Python script. Progress printouts from the script will show the active featureset, data fraction, and split index while AutoGluon is training.


In [ ]:
ensemble_metrics, split_metrics, split_metrics_summary = ml_autogluon_train_test_random(args)


## Inspect returned tables

The script returns three dataframes:
- `ensemble_metrics`: AutoGluon leaderboard-style rows for the models tested
- `split_metrics`: per-split metrics for the selected best model
- `split_metrics_summary`: aggregated summary across splits, including pooled bootstrap metrics when labels are available


In [ ]:
print('ensemble_metrics:', ensemble_metrics.shape)
print('split_metrics:', split_metrics.shape)
print('split_metrics_summary:', split_metrics_summary.shape)


In [ ]:
split_metrics_summary


## Output files

Saved outputs follow the same behavior as the script:
- results are written under `data/ml_prediction/Output/<output_subfolder>/` when `output_subfolder` is non-empty
- otherwise they are written directly under `data/ml_prediction/Output/`
- filenames use `output_fname` as the stem, for example:
  - `<output_fname>_split_metrics.csv`
  - `<output_fname>_split_metrics_summary.csv`
  - `<output_fname>_ensemble_metrics.csv`
